# 04b - Self-Play Training and Promotion

Consume completed 04a games, train a candidate, and run CPU promotion matches. GPU is recommended for gradient updates. The champion changes only after successful promotion.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT, "strategy.yaml")
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Load Inputs

Run 04a for the next iteration first. A missing or changed generation manifest stops training.

In [ ]:
from chess_rl.research_workflow import self_play_inputs
initial_checkpoint, dataset_manifest = self_play_inputs(PROJECT_ROOT, cfg)
print("Frozen initialization:", initial_checkpoint)
print("Settings:", cfg["self_play"])

## Train and Promote One Iteration

Resume optimizer updates from complete checkpoints. Promotion retains the established score, paired-confidence and runtime-failure criteria.

In [ ]:
from chess_rl.self_play import train_and_promote
champion = train_and_promote(PROJECT_ROOT, cfg, initial_checkpoint, dataset_manifest)
print("Retained champion:", champion)
league = read_json(PROJECT_ROOT / "checkpoints/self_play" / cfg["run_id"] / "league.json")
print("Completed iterations:", league["completed_iterations"], "/", cfg["self_play"]["iterations"])

## Continue the League

Repeat 04a then 04b until all configured iterations finish. This optional runner performs that same alternating sequence for the remaining iterations.

In [ ]:
RUN_REMAINING_ITERATIONS = False
if RUN_REMAINING_ITERATIONS:
    from chess_rl.self_play import run_league
    champion = run_league(PROJECT_ROOT, cfg, initial_checkpoint, dataset_manifest)
print("Next: 05a after the chosen training runs have completed.")